In [2]:
# app.py
import os 
import tempfile 
import traceback 
from typing import List, Optional, Dict, Any 
import uvicorn 
from fastapi import FastAPI, HTTPException, UploadFile, File, Form 
from pydantic import BaseModel 
from fastapi.middleware.cors import CORSMiddleware

# ---- 引入 facefusion ----
from facefusion import state_manager, logger
from facefusion.args import apply_args
from facefusion.jobs import job_manager, job_store
from facefusion.setup_args import setup_args
from facefusion.core import process_headless_v2


# =============== FastAPI ===============
app = FastAPI(title="FaceFusion API")


# -------- 请求模型 --------
class RunRequest(BaseModel):
    source_paths: List[str]
    target_path: str
    output_path: str
    options: Optional[Dict[str, Any]] = Field(default_factory=dict)


# -------- 启动初始化 --------
@app.on_event("startup")
def startup():
    print("🔥 Initializing facefusion ...")

    job_keys = ['config_path', 'temp_path', 'jobs_path', 'execution_device_id', 'execution_providers', 'execution_thread_count', 'execution_queue_count', 'download_providers', 'video_memory_strategy', 'system_memory_limit', 'log_level', 'config_path', 'temp_path', 'jobs_path', 'execution_device_id', 'execution_providers', 'execution_thread_count', 'execution_queue_count', 'download_providers', 'video_memory_strategy', 'system_memory_limit', 'log_level', 'config_path', 'temp_path', 'jobs_path', 'source_pattern', 'target_pattern', 'output_pattern', 'execution_device_id', 'execution_providers', 'execution_thread_count', 'execution_queue_count', 'download_providers', 'video_memory_strategy', 'system_memory_limit', 'log_level', 'download_providers', 'download_scope', 'log_level', 'temp_path', 'execution_device_id', 'execution_providers', 'execution_thread_count', 'execution_queue_count', 'download_providers', 'video_memory_strategy', 'system_memory_limit', 'log_level', 'jobs_path', 'log_level', 'job_id', 'jobs_path', 'log_level', 'job_id', 'jobs_path', 'log_level', 'jobs_path', 'log_level', 'halt_on_error', 'job_id', 'jobs_path', 'log_level', 'jobs_path', 'log_level', 'halt_on_error', 'job_id', 'config_path', 'jobs_path', 'log_level', 'job_id', 'config_path', 'jobs_path', 'log_level', 'job_id', 'config_path', 'jobs_path', 'log_level', 'job_id', 'jobs_path', 'log_level', 'job_id', 'config_path', 'temp_path', 'jobs_path', 'execution_device_id', 'execution_providers', 'execution_thread_count', 'execution_queue_count', 'download_providers', 'video_memory_strategy', 'system_memory_limit', 'log_level', 'config_path', 'temp_path', 'jobs_path', 'execution_device_id', 'execution_providers', 'execution_thread_count', 'execution_queue_count', 'download_providers', 'video_memory_strategy', 'system_memory_limit', 'log_level', 'halt_on_error', 'job_id', 'config_path', 'temp_path', 'jobs_path', 'execution_device_id', 'execution_providers', 'execution_thread_count', 'execution_queue_count', 'download_providers', 'video_memory_strategy', 'system_memory_limit', 'log_level', 'config_path', 'temp_path', 'jobs_path', 'execution_device_id', 'execution_providers', 'execution_thread_count', 'execution_queue_count', 'download_providers', 'video_memory_strategy', 'system_memory_limit', 'log_level', 'halt_on_error']
    step_keys = ['source_paths', 'target_path', 'output_path', 'face_detector_model', 'face_detector_angles', 'face_detector_size', 'face_detector_score', 'face_landmarker_model', 'face_landmarker_score', 'face_selector_mode', 'face_selector_order', 'face_selector_gender', 'face_selector_race', 'face_selector_age_start', 'face_selector_age_end', 'reference_face_position', 'reference_face_distance', 'reference_frame_number', 'face_occluder_model', 'face_parser_model', 'face_mask_types', 'face_mask_areas', 'face_mask_regions', 'face_mask_blur', 'face_mask_padding', 'trim_frame_start', 'trim_frame_end', 'temp_frame_format', 'keep_temp', 'output_image_quality', 'output_image_resolution', 'output_audio_encoder', 'output_audio_quality', 'output_audio_volume', 'output_video_encoder', 'output_video_preset', 'output_video_quality', 'output_video_resolution', 'output_video_fps', 'processors', 'age_modifier_model', 'age_modifier_direction', 'deep_swapper_model', 'deep_swapper_morph', 'expression_restorer_model', 'expression_restorer_factor', 'face_debugger_items', 'face_editor_model', 'face_editor_eyebrow_direction', 'face_editor_eye_gaze_horizontal', 'face_editor_eye_gaze_vertical', 'face_editor_eye_open_ratio', 'face_editor_lip_open_ratio', 'face_editor_mouth_grim', 'face_editor_mouth_pout', 'face_editor_mouth_purse', 'face_editor_mouth_smile', 'face_editor_mouth_position_horizontal', 'face_editor_mouth_position_vertical', 'face_editor_head_pitch', 'face_editor_head_yaw', 'face_editor_head_roll', 'face_enhancer_model', 'face_enhancer_blend', 'face_enhancer_weight', 'face_swapper_model', 'face_swapper_pixel_boost', 'frame_colorizer_model', 'frame_colorizer_blend', 'frame_colorizer_size', 'frame_enhancer_model', 'frame_enhancer_blend', 'lip_syncer_model', 'lip_syncer_weight', 'source_paths', 'target_path', 'output_path', 'face_detector_model', 'face_detector_angles', 'face_detector_size', 'face_detector_score', 'face_landmarker_model', 'face_landmarker_score', 'face_selector_mode', 'face_selector_order', 'face_selector_gender', 'face_selector_race', 'face_selector_age_start', 'face_selector_age_end', 'reference_face_position', 'reference_face_distance', 'reference_frame_number', 'face_occluder_model', 'face_parser_model', 'face_mask_types', 'face_mask_areas', 'face_mask_regions', 'face_mask_blur', 'face_mask_padding', 'trim_frame_start', 'trim_frame_end', 'temp_frame_format', 'keep_temp', 'output_image_quality', 'output_image_resolution', 'output_audio_encoder', 'output_audio_quality', 'output_audio_volume', 'output_video_encoder', 'output_video_preset', 'output_video_quality', 'output_video_resolution', 'output_video_fps', 'processors', 'age_modifier_model', 'age_modifier_direction', 'deep_swapper_model', 'deep_swapper_morph', 'expression_restorer_model', 'expression_restorer_factor', 'face_debugger_items', 'face_editor_model', 'face_editor_eyebrow_direction', 'face_editor_eye_gaze_horizontal', 'face_editor_eye_gaze_vertical', 'face_editor_eye_open_ratio', 'face_editor_lip_open_ratio', 'face_editor_mouth_grim', 'face_editor_mouth_pout', 'face_editor_mouth_purse', 'face_editor_mouth_smile', 'face_editor_mouth_position_horizontal', 'face_editor_mouth_position_vertical', 'face_editor_head_pitch', 'face_editor_head_yaw', 'face_editor_head_roll', 'face_enhancer_model', 'face_enhancer_blend', 'face_enhancer_weight', 'face_swapper_model', 'face_swapper_pixel_boost', 'frame_colorizer_model', 'frame_colorizer_blend', 'frame_colorizer_size', 'frame_enhancer_model', 'frame_enhancer_blend', 'lip_syncer_model', 'lip_syncer_weight', 'face_detector_model', 'face_detector_angles', 'face_detector_size', 'face_detector_score', 'face_landmarker_model', 'face_landmarker_score', 'face_selector_mode', 'face_selector_order', 'face_selector_gender', 'face_selector_race', 'face_selector_age_start', 'face_selector_age_end', 'reference_face_position', 'reference_face_distance', 'reference_frame_number', 'face_occluder_model', 'face_parser_model', 'face_mask_types', 'face_mask_areas', 'face_mask_regions', 'face_mask_blur', 'face_mask_padding', 'trim_frame_start', 'trim_frame_end', 'temp_frame_format', 'keep_temp', 'output_image_quality', 'output_image_resolution', 'output_audio_encoder', 'output_audio_quality', 'output_audio_volume', 'output_video_encoder', 'output_video_preset', 'output_video_quality', 'output_video_resolution', 'output_video_fps', 'processors', 'age_modifier_model', 'age_modifier_direction', 'deep_swapper_model', 'deep_swapper_morph', 'expression_restorer_model', 'expression_restorer_factor', 'face_debugger_items', 'face_editor_model', 'face_editor_eyebrow_direction', 'face_editor_eye_gaze_horizontal', 'face_editor_eye_gaze_vertical', 'face_editor_eye_open_ratio', 'face_editor_lip_open_ratio', 'face_editor_mouth_grim', 'face_editor_mouth_pout', 'face_editor_mouth_purse', 'face_editor_mouth_smile', 'face_editor_mouth_position_horizontal', 'face_editor_mouth_position_vertical', 'face_editor_head_pitch', 'face_editor_head_yaw', 'face_editor_head_roll', 'face_enhancer_model', 'face_enhancer_blend', 'face_enhancer_weight', 'face_swapper_model', 'face_swapper_pixel_boost', 'frame_colorizer_model', 'frame_colorizer_blend', 'frame_colorizer_size', 'frame_enhancer_model', 'frame_enhancer_blend', 'lip_syncer_model', 'lip_syncer_weight', 'face_detector_model', 'face_detector_angles', 'face_detector_size', 'face_detector_score', 'face_landmarker_model', 'face_landmarker_score', 'face_selector_mode', 'face_selector_order', 'face_selector_gender', 'face_selector_race', 'face_selector_age_start', 'face_selector_age_end', 'reference_face_position', 'reference_face_distance', 'reference_frame_number', 'face_occluder_model', 'face_parser_model', 'face_mask_types', 'face_mask_areas', 'face_mask_regions', 'face_mask_blur', 'face_mask_padding', 'trim_frame_start', 'trim_frame_end', 'temp_frame_format', 'keep_temp', 'output_image_quality', 'output_image_resolution', 'output_audio_encoder', 'output_audio_quality', 'output_audio_volume', 'output_video_encoder', 'output_video_preset', 'output_video_quality', 'output_video_resolution', 'output_video_fps', 'processors', 'age_modifier_model', 'age_modifier_direction', 'deep_swapper_model', 'deep_swapper_morph', 'expression_restorer_model', 'expression_restorer_factor', 'face_debugger_items', 'face_editor_model', 'face_editor_eyebrow_direction', 'face_editor_eye_gaze_horizontal', 'face_editor_eye_gaze_vertical', 'face_editor_eye_open_ratio', 'face_editor_lip_open_ratio', 'face_editor_mouth_grim', 'face_editor_mouth_pout', 'face_editor_mouth_purse', 'face_editor_mouth_smile', 'face_editor_mouth_position_horizontal', 'face_editor_mouth_position_vertical', 'face_editor_head_pitch', 'face_editor_head_yaw', 'face_editor_head_roll', 'face_enhancer_model', 'face_enhancer_blend', 'face_enhancer_weight', 'face_swapper_model', 'face_swapper_pixel_boost', 'frame_colorizer_model', 'frame_colorizer_blend', 'frame_colorizer_size', 'frame_enhancer_model', 'frame_enhancer_blend', 'lip_syncer_model', 'lip_syncer_weight', 'source_paths', 'target_path', 'output_path', 'face_detector_model', 'face_detector_angles', 'face_detector_size', 'face_detector_score', 'face_landmarker_model', 'face_landmarker_score', 'face_selector_mode', 'face_selector_order', 'face_selector_gender', 'face_selector_race', 'face_selector_age_start', 'face_selector_age_end', 'reference_face_position', 'reference_face_distance', 'reference_frame_number', 'face_occluder_model', 'face_parser_model', 'face_mask_types', 'face_mask_areas', 'face_mask_regions', 'face_mask_blur', 'face_mask_padding', 'trim_frame_start', 'trim_frame_end', 'temp_frame_format', 'keep_temp', 'output_image_quality', 'output_image_resolution', 'output_audio_encoder', 'output_audio_quality', 'output_audio_volume', 'output_video_encoder', 'output_video_preset', 'output_video_quality', 'output_video_resolution', 'output_video_fps', 'processors', 'age_modifier_model', 'age_modifier_direction', 'deep_swapper_model', 'deep_swapper_morph', 'expression_restorer_model', 'expression_restorer_factor', 'face_debugger_items', 'face_editor_model', 'face_editor_eyebrow_direction', 'face_editor_eye_gaze_horizontal', 'face_editor_eye_gaze_vertical', 'face_editor_eye_open_ratio', 'face_editor_lip_open_ratio', 'face_editor_mouth_grim', 'face_editor_mouth_pout', 'face_editor_mouth_purse', 'face_editor_mouth_smile', 'face_editor_mouth_position_horizontal', 'face_editor_mouth_position_vertical', 'face_editor_head_pitch', 'face_editor_head_yaw', 'face_editor_head_roll', 'face_enhancer_model', 'face_enhancer_blend', 'face_enhancer_weight', 'face_swapper_model', 'face_swapper_pixel_boost', 'frame_colorizer_model', 'frame_colorizer_blend', 'frame_colorizer_size', 'frame_enhancer_model', 'frame_enhancer_blend', 'lip_syncer_model', 'lip_syncer_weight', 'source_paths', 'output_path', 'face_detector_model', 'face_detector_angles', 'face_detector_size', 'face_detector_score', 'face_landmarker_model', 'face_landmarker_score', 'face_selector_mode', 'face_selector_order', 'face_selector_gender', 'face_selector_race', 'face_selector_age_start', 'face_selector_age_end', 'reference_face_position', 'reference_face_distance', 'reference_frame_number', 'face_occluder_model', 'face_parser_model', 'face_mask_types', 'face_mask_areas', 'face_mask_regions', 'face_mask_blur', 'face_mask_padding', 'trim_frame_start', 'trim_frame_end', 'temp_frame_format', 'keep_temp', 'output_image_quality', 'output_image_resolution', 'output_audio_encoder', 'output_audio_quality', 'output_audio_volume', 'output_video_encoder', 'output_video_preset', 'output_video_quality', 'output_video_resolution', 'output_video_fps', 'processors', 'age_modifier_model', 'age_modifier_direction', 'deep_swapper_model', 'deep_swapper_morph', 'expression_restorer_model', 'expression_restorer_factor', 'face_debugger_items', 'face_editor_model', 'face_editor_eyebrow_direction', 'face_editor_eye_gaze_horizontal', 'face_editor_eye_gaze_vertical', 'face_editor_eye_open_ratio', 'face_editor_lip_open_ratio', 'face_editor_mouth_grim', 'face_editor_mouth_pout', 'face_editor_mouth_purse', 'face_editor_mouth_smile', 'face_editor_mouth_position_horizontal', 'face_editor_mouth_position_vertical', 'face_editor_head_pitch', 'face_editor_head_yaw', 'face_editor_head_roll', 'face_enhancer_model', 'face_enhancer_blend', 'face_enhancer_weight', 'face_swapper_model', 'face_swapper_pixel_boost', 'frame_colorizer_model', 'frame_colorizer_blend', 'frame_colorizer_size', 'frame_enhancer_model', 'frame_enhancer_blend', 'lip_syncer_model', 'lip_syncer_weight', 'source_paths', 'target_path', 'output_path', 'face_detector_model', 'face_detector_angles', 'face_detector_size', 'face_detector_score', 'face_landmarker_model', 'face_landmarker_score', 'face_selector_mode', 'face_selector_order', 'face_selector_gender', 'face_selector_race', 'face_selector_age_start', 'face_selector_age_end', 'reference_face_position', 'reference_face_distance', 'reference_frame_number', 'face_occluder_model', 'face_parser_model', 'face_mask_types', 'face_mask_areas', 'face_mask_regions', 'face_mask_blur', 'face_mask_padding', 'trim_frame_start', 'trim_frame_end', 'temp_frame_format', 'keep_temp', 'output_image_quality', 'output_image_resolution', 'output_audio_encoder', 'output_audio_quality', 'output_audio_volume', 'output_video_encoder', 'output_video_preset', 'output_video_quality', 'output_video_resolution', 'output_video_fps', 'processors', 'age_modifier_model', 'age_modifier_direction', 'deep_swapper_model', 'deep_swapper_morph', 'expression_restorer_model', 'expression_restorer_factor', 'face_debugger_items', 'face_editor_model', 'face_editor_eyebrow_direction', 'face_editor_eye_gaze_horizontal', 'face_editor_eye_gaze_vertical', 'face_editor_eye_open_ratio', 'face_editor_lip_open_ratio', 'face_editor_mouth_grim', 'face_editor_mouth_pout', 'face_editor_mouth_purse', 'face_editor_mouth_smile', 'face_editor_mouth_position_horizontal', 'face_editor_mouth_position_vertical', 'face_editor_head_pitch', 'face_editor_head_yaw', 'face_editor_head_roll', 'face_enhancer_model', 'face_enhancer_blend', 'face_enhancer_weight', 'face_swapper_model', 'face_swapper_pixel_boost', 'frame_colorizer_model', 'frame_colorizer_blend', 'frame_colorizer_size', 'frame_enhancer_model', 'frame_enhancer_blend', 'lip_syncer_model', 'lip_syncer_weight']

    try:
        job_store.register_job_keys(job_keys)
        job_store.register_step_keys(step_keys)
    except:
        pass

    defaults = setup_args()
    try:
        apply_args(defaults, state_manager.init_item)
    except:
        pass

    try:
        logger.init(state_manager.get_item("log_level"))
    except:
        logger.init("info")

    try:
        job_manager.init_jobs(state_manager.get_item("jobs_path"))
    except:
        default_path = os.path.abspath("./jobs")
        os.makedirs(default_path, exist_ok=True)
        state_manager.init_item("jobs_path", default_path)
        job_manager.init_jobs(default_path)

    print("✅ FaceFusion Initialized!")


# -------- 执行接口 --------
@app.post("/run")
async def run_job(
    source_file: UploadFile = File(..., description="source 人脸图片（png/jpg）"),
    target_file: UploadFile = File(..., description="target 图片/视频（png/jpg）"),
    output_format: str = Form("jpg"),
    options: Optional[str] = Form(None),
):
    """
    接受图片文件上传（source 为单个文件），限制 PNG/JPG
    """

    # ---- 校验文件类型 ----
    validate_image_file(source_file, "source_file")
    validate_image_file(target_file, "target_file")

    # ---- 保存上传文件到 temp 路径 ----
    temp_dir =state_manager.get_item("temp_path")
    os.makedirs(temp_dir, exist_ok=True)

    # 保存 source
    source_path = os.path.join(temp_dir, source_file.filename)
    with open(source_path, "wb") as fp:
        fp.write(await source_file.read())

    # 保存 target
    target_path = os.path.join(temp_dir, target_file.filename)
    with open(target_path, "wb") as fp:
        fp.write(await target_file.read())

    # 输出文件
    output_path = os.path.join(temp_dir, f"output.{output_format}")

    # ---- 创建 args ----
    args = setup_args()
    args["command"] = "headless-run"
    args["source_paths"] = [source_path]
    args["target_path"] = target_path
    args["output_path"] = output_path

    # ---- options JSON ----
    import json
    if options:
        try:
            opts = json.loads(options)
            args.update(opts)
        except:
            raise HTTPException(400, "options 必须是 JSON 字符串")

    try:
        apply_args(args, state_manager.init_item)
        result = process_headless_v2(args)

        if os.path.exists(output_path):
            with open(output_path, "rb") as fp:
                data = fp.read()
            return {
                "ok": True,
                "output_path": output_path,
                "output_size": len(data),
                "result": result,
            }
        else:
            return {"ok": False, "error": "output not generated"}

    except Exception as e:
        return {
            "ok": False,
            "error": str(e),
            "traceback": traceback.format_exc(),
        }

@app.get("/health")
def health():
    return {"status": "ok"}


# -------- 主入口，等价于 uvicorn app:app --------
if __name__ == "__main__":
    print("🚀 Starting FastAPI server on 0.0.0.0:6006 ...")
    uvicorn.run(
        "app:app",   # 这里一定要字符串写法，否则 reload 不支持
        host="0.0.0.0",
        port=6006,
        reload=False    # 如果你需要自动重载可以改成 True
    )


🚀 Starting FastAPI server on 0.0.0.0:6006 ...


/tmp/ipykernel_944/1980610786.py:32: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("startup")


RuntimeError: asyncio.run() cannot be called from a running event loop